# Bermudan Callable Floater Valuation

In this notebook, we demonstrate the valuation of a Bermudan callable floater.

The callable Floater is decomposed into the non-callable Floater with contractual final maturity and the Bermudan option to enter into a floater with offsetting cash flows.

Critical point for modelling is taking into account credit risk of the floater. We use a credit-hybrid model. The credit model component is modelled as stochastic hazard rate model with CIR-like model volatility.

In [ ]:
using Pkg
Pkg.activate("../.")

In [ ]:
using CSV
using DataFrames
using DiffFusion
using Plots
using StatsBase

## Model Specification

In this section, we specify the valuation model.

### Correlations

For this example, we use independent modelling of interest rates and hazard rates.

Correlation between rates and credit can be modelled by setting the elements of the correlation holder.

In [ ]:
ch = DiffFusion.correlation_holder("ch/One")

### Interest Rate Model

Interest rates are modelled via a one-factor Gaussian model. We use flat model parameters to keep the example brief.

In [ ]:
delta = DiffFusion.flat_parameter(0.0)
chi = DiffFusion.flat_parameter(0.01)
sigma = DiffFusion.flat_volatility(0.01)

mdl_eur = DiffFusion.gaussian_hjm_model(
    "mdl/EUR",
    delta,
    chi,
    sigma,
    ch,
    nothing,  # quanto model
    DiffFusion.DiagonalScaling,
)

### Credit Hazard Rate Model

We use a hazard rate model with volatility function similar to a CIR model.

The hazard rate model uses our quasi-Gaussian model framework. The quasi-Gaussian model short rate is interpreted and used as hazard rate.

In the quasi-Gaussian model setting, the volatility is formulated in terms the state variable $x_t$. In the CIR model, the volatility is formulated in terms of the short rate $r_t$.

We use the initial forward rate curve $f(0,t)$ and the relation $r_t = f(0,t) + x_t$ to formulate the quasi-Gaussian model volatility as
$$
  \sigma(t) \sqrt{x_t + f(0,t)}.
$$
Here, $\sigma(t)$ is a (piece-wise) constant volatility parameter.

The model ensures, that (up to numerical discretisation errors) the hazard rate remains non-negative if the initial forward rates are non-negative.

We assume an input (CDS) credit spread of 500bp and a recovery rate of 40%.

The parametrisation is translated into an initial hazard rate using the approximation in Brigo/Mercurio (2007), section 21.3.6.

In [ ]:
ts_credit = DiffFusion.flat_forward("ts/Credit", 0.05 / (1.0 - 0.4))  # See Brigo approximation
min_short_rate = 1.0e-8
volatility_function = DiffFusion.CirShortRateModelFunction(ts_credit, min_short_rate)

The credit model is further parametrised with 10% mmean reversion rate and 30% volatility parameter.

In [ ]:
chi_cir = DiffFusion.flat_parameter(0.10)
sigma_cir = DiffFusion.flat_volatility(0.30)

mdl_credit = DiffFusion.quasi_gaussian_short_rate_model(
    "mdl/Credit",
    chi_cir,
    sigma_cir,
    nothing,  # no quanto model
    volatility_function,
)

### Hybrid Model

Interest rate model and credit model are combined into a credit-hybrid model.

In [ ]:
mdl_hybrid = DiffFusion.diagonal_model("mdl/Hybrid", [mdl_eur, mdl_credit])

## Monte Carlo Simulation

We simulate the model state variables.

Note that quasi-Gaussian model simulation is considerably more time consuming than Gaussian model simulation. As a consequence, we use rather coarse numerical discretisation parameters.

In [ ]:
times = 0.0:1.0/4:10.0
n_paths = 2^10
sim = DiffFusion.state_dependent_simulation(
    mdl_hybrid,
    ch,
    times,
    n_paths,
    with_progress_bar = true,  # not really working in VS Code
    brownian_increments = DiffFusion.sobol_brownian_increments,
)

println("Simulation completed with $(size(sim.X)) states.")

## Valuation Context and Path

We specify the valuation context and *path* object for payoff and product valuation.

In [ ]:
ts_estr = DiffFusion.flat_forward("ts/Estr", 0.0300)
ts_euribor3m = DiffFusion.flat_forward("ts/Euribor3m", 0.0350)

ts_list = [
    ts_estr,
    ts_euribor3m,
    ts_credit,
]

ctx = DiffFusion.context(
    "Std",
    DiffFusion.numeraire_entry("EUR", "mdl/EUR", "ts/Estr"),
    [ 
        DiffFusion.rates_entry("EUR", "mdl/EUR", Dict("ESTR" => "ts/Estr", "EURIBOR3M" => "ts/Euribor3m")),
        DiffFusion.rates_entry("EU_CORP", "mdl/Credit", "ts/Credit"), 
    ],
)

path = DiffFusion.path(
    sim,
    ts_list,
    ctx,
    DiffFusion.LinearPathInterpolation,
)

Additionally, we specify a deterministic simulation model.

The deterministic model is used to calculate the non-callable Floater components. 

In [ ]:
sim_0 = DiffFusion.state_dependent_simulation(
    mdl_hybrid,
    ch,
    [0.0],  # no time steps, only initial state
    1,      # only one path
    with_progress_bar = false,
    brownian_increments = DiffFusion.sobol_brownian_increments,
)

ctx_0 = DiffFusion.context(  # no models linked
    "Std",
    DiffFusion.numeraire_entry("EUR", nothing, "ts/Estr"),
    [ 
        DiffFusion.rates_entry("EUR", nothing, Dict("ESTR" => "ts/Estr", "EURIBOR3M" => "ts/Euribor3m")),
        DiffFusion.rates_entry("EU_CORP", nothing, "ts/Credit"), 
    ],
)

path_0 = DiffFusion.path(
    sim_0,
    ts_list,
    ctx_0,
    DiffFusion.LinearPathInterpolation,
)


## Callable Floater Product Setup

In this section, we specify the floater cash flows and exercise events.

### Cash Flow Legs

We specify the cash flow legs if option is exercised.

In [ ]:
function exercise_legs(
    exercise_time,
    coupon_schedule,
    spread_rate,
    recovery_rate,
    )
    times = [ t for t in coupon_schedule if t > exercise_time ]
    #
    coupons = DiffFusion.CashFlow[
        DiffFusion.SimpleRateCoupon(s, s, e, e, e - s, "EUR:EURIBOR3M", nothing, spread_rate)
        for (s, e) in zip(times[1:end-1], times[2:end])
    ]
    euribor_flows = DiffFusion.CashFlow[
        DiffFusion.SimpleRateCoupon(s, s, e, e, e - s, "EUR:EURIBOR3M", nothing, 0.0)
        for (s, e) in zip(times[1:end-1], times[2:end])
    ]
    annuity_flows = DiffFusion.CashFlow[
        DiffFusion.FixedRateCoupon(e, 1.0, e - s, s)
        for (s, e) in zip(times[1:end-1], times[2:end])
    ]
    #
    strike = DiffFusion.FixedCashFlow(times[begin], -1.0)  # make sure after exercise time
    redemption = DiffFusion.FixedCashFlow(times[end], 1.0)
    #
    floater_leg = DiffFusion.cashflow_leg(
        "leg/Floater",
        vcat(strike, coupons, redemption),
        100.0,
        "EUR:ESTR",
        nothing,
        1.0,  # receive coupons/notional
    )
    euribor_leg = DiffFusion.cashflow_leg(
        "leg/Euribor",
        vcat(strike, euribor_flows, redemption),
        100.0,
        "EUR:ESTR",
        nothing,
        -1.0,  # payer for positive leg npv
    )
    annuity_leg = DiffFusion.cashflow_leg(
        "leg/Annuity",
        annuity_flows,
        100.0,
        "EUR:ESTR",
        nothing,
        1.0, #  receiver for positive leg npv
    )
    #
    floater_leg = DiffFusion.credit_risky_cashflow_leg(floater_leg, "EU_CORP", recovery_rate)
    euribor_leg = DiffFusion.credit_risky_cashflow_leg(euribor_leg, "EU_CORP", recovery_rate)
    annuity_leg = DiffFusion.credit_risky_cashflow_leg(annuity_leg, "EU_CORP", 0.0)  # no redemption flows here
    #
    return floater_leg, euribor_leg, annuity_leg
end


### Valuation Functions via Payoffs

European options and option components are calculated by generating *discounted cash flow payoffs* at exercise time.

The resulting payoffs are then applied to the Monte-Carlo simulation to calculate a *model price*.

In [ ]:
function european_option_valuation(
    exercise_time,
    coupon_schedule,
    spread_rate,
    recovery_rate,
    )
    floater_leg, euribor_leg, annuity_leg = exercise_legs(exercise_time, coupon_schedule, spread_rate, recovery_rate)
    floater_leg_payoffs = DiffFusion.discounted_cashflows(floater_leg, exercise_time)
    option_payoff = DiffFusion.Max(0.0, sum(floater_leg_payoffs))
    #
    npv_option = DiffFusion.model_price(
        [option_payoff],
        path,
        exercise_time,
        "EUR",
    )
    npv_floater_leg = DiffFusion.model_price(
        floater_leg_payoffs,
        path,
        exercise_time,
        "EUR",
    )
    npv_euribor_leg = DiffFusion.model_price(
        DiffFusion.discounted_cashflows(euribor_leg, exercise_time),
        path,
        exercise_time,
        "EUR",
    )
    npv_annuity_leg = DiffFusion.model_price(
        DiffFusion.discounted_cashflows(annuity_leg, exercise_time),
        path,
        exercise_time,
        "EUR",
    )
    return npv_option, npv_floater_leg, npv_euribor_leg, npv_annuity_leg
end


Bermudan option valuation requires setting up American Monte-Carlo payoffs. This is implemented by means of `BermudanExercise` objects.

The `BermudanSwaptionLeg` implements the backward induction algorithm in terms of specified payoffs.

The initial hold value of the Bermudan $H_0$ is represented as a payoff within the `BermudanSwaptionLeg` object. The payoff for $H_0$ is then applied to the Monte-Carlo simulation to calculate the Bermudan price.

In [ ]:
function bermudan_option_valuation(
    exercise_times,
    coupon_schedule,
    spread_rate,
    recovery_rate,
    )
    #
    make_regression_variables(t) = [
        DiffFusion.LiborRate(t, t, coupon_schedule[end], "EUR:EURIBOR3M"),
        DiffFusion.LiborRate(t, t, coupon_schedule[end], "EU_CORP"),  # a credit rate
        DiffFusion.Max(
            0.0,
            spread_rate - DiffFusion.LiborRate(t, t, coupon_schedule[end], "EU_CORP")
        ),
    ]
    # make_regression = (C, O) -> DiffFusion.polynomial_regression(C, O, 2)
    make_regression = (C, O) -> DiffFusion.piecewise_regression(C, O, 2, [1, 3, 3])
    #
    exercises = DiffFusion.BermudanExercise[]
    for exercise_time in exercise_times
        floater_leg, _, _ = exercise_legs(exercise_time, coupon_schedule, spread_rate, recovery_rate)
        exercise = DiffFusion.bermudan_exercise(
            exercise_time,
            [floater_leg],
            make_regression_variables
        )
        push!(exercises, exercise)
    end
    #
    berm = DiffFusion.bermudan_swaption_leg(
        "berm/floater",
        exercises,
        1.0, # long option
        "EUR", # default discounting
        make_regression_variables,
        nothing, # path
        nothing, # make_regression
    )
    DiffFusion.reset_regression!(berm, path, make_regression)
    #
    berm_payoff = berm.hold_values[1]
    berm_npv = DiffFusion.model_price(
        [berm_payoff],
        path,
        DiffFusion.obs_time(berm_payoff),
        "EUR"
    )
    return berm_npv
end


### Fair Spread Rate Scenarios

We also implement a function to calculate future simulated fair funding spread rates.

In [ ]:
function spread_rate_scenarios(
    exercise_time,
    coupon_schedule,
    spread_rate,
    recovery_rate,
    path,
    )
    _, euribor_leg, annuity_leg = exercise_legs(exercise_time, coupon_schedule, spread_rate, recovery_rate)
    euribor_leg = sum(DiffFusion.discounted_cashflows(euribor_leg, exercise_time))
    annuity_leg = sum(DiffFusion.discounted_cashflows(annuity_leg, exercise_time))
    spread_rate = euribor_leg / annuity_leg
    return spread_rate(path)
end

## Product Valuation

In this section, we specify the actual product and calculate the pricing components.

In [ ]:
effective_time = 0.0
maturity_time = 10.0
term = 0.25
spread_rate = 0.05 # 0.035

recovery_rate = 0.4

coupon_schedule = effective_time:term:maturity_time

floater_leg, euribor_leg, annuity_leg = exercise_legs(-1.0, coupon_schedule, spread_rate, recovery_rate)

npv_floater_leg = DiffFusion.model_price(
    DiffFusion.discounted_cashflows(floater_leg, 0.0),
    path_0,
    0.0,
    "EUR",
)
npv_euribor_leg = DiffFusion.model_price(
    DiffFusion.discounted_cashflows(euribor_leg, 0.0),
    path_0,
    0.0,
    "EUR",
)
npv_annuity_leg = DiffFusion.model_price(
    DiffFusion.discounted_cashflows(annuity_leg, 0.0),
    path_0,
    0.0,
    "EUR",
)

fair_spread_rate = (100.0 + npv_euribor_leg) / npv_annuity_leg

println("Floater Leg NPV: $(npv_floater_leg)")
println("Euribor Leg NPV: $(npv_euribor_leg)")
println("Annuity Leg NPV: $(npv_annuity_leg)")
println("Fair Spread Rate: $(fair_spread_rate)")

Also, we calculate and plot future simulated fair spread rates.

In [ ]:
ε = 1.0 / 365 / 24 / 3600  # 1 second in years

for exercise_time in [1.0, 2.0, 5.0] .- ε
    spread_rates = spread_rate_scenarios(
        exercise_time,
        coupon_schedule,
        spread_rate,
        recovery_rate,
        path,
    )
    spread_vol = std(spread_rates) / sqrt(exercise_time) / mean(spread_rates)
    T = round(exercise_time, digits=1)
    σ = round(spread_vol*100, digits=1)
    fig = histogram(
        spread_rates,
        bins=50,
        title="Spread Rate Scenarios, T=$(T)y, σ=$(σ)%",
        xlabel="Spread Rate",
        ylabel="Frequency",
    )
    display(fig)
end


We price the European and Bermudan option for various spreads to model different moneyness levels.

In [ ]:
ε = 1.0 / 365 / 24 / 3600  # 1 second in years
exersise_times = (1.0:1.0:9.0) .- ε  # exercise just before the coupon date

result_table = DataFrame()
for spread_rate in [0.01, 0.03, 0.10]  # 0.01:0.01:0.10
    results = []
    for exercise_time in exersise_times
        local term, npv_option, npv_floater_leg, npv_euribor_leg, npv_annuity_leg
        expiry = Int(round(exercise_time, digits=0))
        term   = Int(round(coupon_schedule[end] - exercise_time, digits=0))
        npv_option, npv_floater_leg, npv_euribor_leg, npv_annuity_leg = european_option_valuation(
            exercise_time,
            coupon_schedule,
            spread_rate,
            recovery_rate,
        )
        result = (
            expiry = "$(expiry)y",
            term = "$(term)y",
            payer_or_receiver = -1.0,  # Payer
            strike_rate = spread_rate,
            fair_rate = npv_euribor_leg / npv_annuity_leg,
            annuity = npv_annuity_leg,
            exercise_time = exercise_time,
            european_npv = npv_option,
        )
        push!(results, result)
    end
    table = DataFrame(results)
    #
    npv_berm = bermudan_option_valuation(
        exersise_times,
        coupon_schedule,
        spread_rate,
        recovery_rate,
    )
    table[!, :bermudan_npv] .= npv_berm
    #
    println(table)
    global result_table = vcat(result_table, table)
end

Finally, we plot European and Bermudan options

In [ ]:
for strike_rate in unique(result_table.strike_rate)
    table = filter(row -> row.strike_rate == strike_rate, result_table)
    x = [ e * "-" * s for (e, s) in zip(table.expiry, table.term) ]
    y = table.european_npv
    #
    push!(x, "Berm")
    push!(y, table.bermudan_npv[begin])
    fig = bar(
        x,
        y,
        title="European and Bermudan Options, spread rate $(strike_rate*100)%",
        xlabel="Option",
        ylabel="Price",
        legend=false,
    )
    display(fig)
end

In general, we see the expected distribution of option component prices across expiry times.

Interestingly, for funding spreads equal to 1%, where the option to call the floater is far out-of-the-money, the European option profile is not concave. Instead, the very first European option is considerably more expensive than the second European option.

Note, the unusual out-of-the-money option profile is a numerical artifact of the coarse Monte-Carlo time discretisation with quarterly time steps. Changing, e.g., to monthly time steps leads to the concave profile in line with expectations. This observation highlights the need to closely pay attention to the numerical simulation properties when simulating Quasi-Gaussian models.
